# Qualcomm Live Coding Prep
**Amsterdam AI Research — Senior MLE / Physical AI**

Practical Python + NumPy/PyTorch drills (not LeetCode). Interviewers care about correctness, edge cases, numerical stability, and clear narration.

**How to drill each prompt (25–40 min)**
1. Restate the problem and constraints out loud
2. Write a clear signature + docstring
3. Implement the happy path
4. Handle zeros / empties / shapes / dtypes
5. Run the unit-test cell below it (real `pytest`, one named test per behaviour)
6. Discuss complexity, vectorization, and follow-ups

**Test convention:** every test cell starts with `%%ipytest -qq` and contains `test_*` functions,
so failures are reported per behaviour instead of stopping at the first `assert`.
Fixtures and `@pytest.mark.parametrize` are used where they earn their keep — talking through
that structure is itself good interview material.

Select kernel: **Qualcomm Live Coding (Python 3.13)**


## Setup


In [1]:
import copy
import math
import random
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import ipytest
import pytest

ipytest.autoconfig()

print(f"numpy {np.__version__}")
print(f"torch {torch.__version__}")
print(f"pytest {pytest.__version__}")
print("python ready")


numpy 2.5.2
torch 2.13.0
pytest 9.1.1
python ready


---
## Warm-up (already practiced): Precision / Recall / F1

Rewrite cold. Watch boolean ops, zero-division, and unused code.


In [2]:
import numpy as np
def precision_recall_f1(y_true, y_pred):
    """y_true and y_pred are lists/arrays of 0/1 integers. Return (precision, recall, f1)."""
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = (y_true * y_pred).sum()
    fp = ((1 - y_true) * y_pred).sum()
    tn = ((1 - y_true) * (1 - y_pred)).sum()
    fn = (y_true * (1 - y_pred)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall)  if  (precision + recall)  > 0 else 0.0

    return precision, recall, f1



In [3]:
%%ipytest -qq
# Unit tests — warm-up: precision / recall / f1


@pytest.mark.parametrize(
    "y_true, y_pred, expected",
    [
        ([1, 1, 0, 0, 1], [1, 0, 0, 1, 1], (2 / 3, 2 / 3, 2 / 3)),
        ([1, 1, 1, 1], [1, 1, 1, 1], (1.0, 1.0, 1.0)),
        ([1, 0], [0, 1], (0.0, 0.0, 0.0)),
    ],
)
def test_known_values(y_true, y_pred, expected):
    assert precision_recall_f1(y_true, y_pred) == pytest.approx(expected)


def test_no_positive_predictions_returns_zeros():
    assert precision_recall_f1([0, 0, 0], [0, 0, 0]) == (0.0, 0.0, 0.0)


def test_f1_is_harmonic_mean_of_precision_and_recall():
    precision, recall, f1 = precision_recall_f1([1, 1, 0, 1], [1, 0, 1, 1])
    assert f1 == pytest.approx(2 * precision * recall / (precision + recall))


def test_perfect_recall_with_imperfect_precision():
    # every positive found, but one false positive
    precision, recall, _ = precision_recall_f1([1, 1, 0], [1, 1, 1])
    assert recall == pytest.approx(1.0)
    assert precision == pytest.approx(2 / 3)


......                                                                                       [100%]


---
# Tier A — Highest probability (metrics, tensors, ML utils)


### A1. Softmax (stable)
**Watch for:** subtract max along class axis; 1D vs 2D; probs sum to 1.
**Follow-ups:** temperature; log-softmax; overflow — implement the next stubs cell, then run the follow-up tests.


In [4]:
def softmax(logits):
    """logits: 1D or 2D numpy array (batch, classes) if 2D.
    Return probabilities with same shape. Numerically stable.
    """

    logits_exp = np.exp(logits - logits.max(axis=-1, keepdims=True))
    probabilities = logits_exp / logits_exp.sum(axis=-1, keepdims=True)

    return probabilities


In [5]:
%%ipytest -qq
# Unit tests — A1 softmax


@pytest.fixture
def logits_1d():
    return np.array([1.0, 2.0, 3.0])


@pytest.fixture
def logits_2d():
    return np.array([[1000.0, 1000.0], [1.0, 2.0]])


def test_1d_shape_and_normalisation(logits_1d):
    probs = softmax(logits_1d)
    assert probs.shape == logits_1d.shape
    assert probs.sum() == pytest.approx(1.0)
    assert np.argmax(probs) == 2


def test_rows_sum_to_one(logits_2d):
    probs = softmax(logits_2d)
    assert probs.shape == logits_2d.shape
    assert probs.sum(axis=-1) == pytest.approx(np.ones(2))
    assert np.isfinite(probs).all()


def test_matches_torch(logits_1d, logits_2d):
    reference = nn.Softmax(dim=-1)
    for logits in (logits_1d, logits_2d):
        expected = reference(torch.tensor(logits)).numpy()
        assert softmax(logits) == pytest.approx(expected)


def test_is_shift_invariant(logits_1d):
    assert softmax(logits_1d) == pytest.approx(softmax(logits_1d + 100.0))


def test_equal_logits_give_uniform_distribution():
    assert softmax(np.zeros(4)) == pytest.approx(np.full(4, 0.25))


.....                                                                                        [100%]


In [10]:
# A1 follow-ups — implement these (or extend your softmax above)

def softmax_temperature(logits, temperature=1.0):
    """Stable softmax with temperature > 0.
    T=1 -> standard softmax; T>1 softer; 0<T<1 sharper.
    """
    
    temp_logits = logits / temperature
    exp = np.exp(temp_logits - temp_logits.max(axis=-1, keepdims=True))
    
    probabilities = exp / exp.sum(axis=-1, keepdims=True)

    return probabilities


def log_softmax(logits, temperature=1.0):
    """Stable log(softmax(logits / T)). Avoid log(softmax(...)) for numerics."""
    
    temp_logits = logits / temperature
    shifted = temp_logits - temp_logits.max(axis=-1, keepdims=True)
    exp = np.exp(shifted)
    
    log_prob = shifted - np.log(exp.sum(axis=-1, keepdims=True))

    return log_prob

In [11]:
%%ipytest -qq
# Unit tests — A1 follow-ups: overflow, temperature, log-softmax

HUGE = np.array([1e5, 1e5 + 1.0, 1e5 + 2.0])
TINY = np.array([-1e5, -1e5 - 1.0, -1e5 - 2.0])
UNIFORM_3 = np.full(3, 1 / 3)


@pytest.fixture
def logits():
    return np.array([1.0, 2.0, 3.0])


# --- overflow ---
@pytest.mark.parametrize("extreme, expected_argmax", [(HUGE, 2), (TINY, 0)])
def test_softmax_survives_extreme_logits(extreme, expected_argmax):
    probs = softmax(extreme)
    assert np.isfinite(probs).all()
    assert probs.sum() == pytest.approx(1.0)
    assert np.argmax(probs) == expected_argmax


def test_naive_exp_overflows_but_shifted_exp_does_not():
    with np.errstate(over="ignore"):
        assert not np.isfinite(np.exp(HUGE)).all()
    assert np.isfinite(np.exp(HUGE - HUGE.max())).all()


# --- temperature ---
def test_temperature_one_matches_plain_softmax(logits):
    assert softmax_temperature(logits, temperature=1.0) == pytest.approx(softmax(logits))


def test_high_temperature_moves_towards_uniform(logits):
    baseline = softmax_temperature(logits, temperature=1.0)
    softened = softmax_temperature(logits, temperature=10.0)
    assert np.linalg.norm(softened - UNIFORM_3) < np.linalg.norm(baseline - UNIFORM_3)
    assert softmax_temperature(logits, temperature=1e6) == pytest.approx(UNIFORM_3, abs=1e-4)


def test_low_temperature_sharpens_towards_argmax(logits):
    sharpened = softmax_temperature(logits, temperature=0.1)
    assert sharpened[2] > softmax_temperature(logits, temperature=1.0)[2]
    assert sharpened.sum() == pytest.approx(1.0)


def test_temperature_preserves_batch_shape():
    batch = np.array([[1.0, 2.0, 3.0], [3.0, 2.0, 1.0]])
    probs = softmax_temperature(batch, temperature=2.0)
    assert probs.shape == batch.shape
    assert probs.sum(axis=-1) == pytest.approx(np.ones(2))


def test_temperature_leaves_ranking_unchanged(logits):
    for temperature in (0.25, 1.0, 4.0):
        ranking = np.argsort(softmax_temperature(logits, temperature))
        assert ranking.tolist() == np.argsort(logits).tolist()


# --- log-softmax ---
def test_log_softmax_exponentiates_to_softmax(logits):
    log_probs = log_softmax(logits)
    assert log_probs.shape == logits.shape
    assert np.exp(log_probs) == pytest.approx(softmax(logits))


def test_log_softmax_matches_torch():
    batch = np.array([[1.0, 2.0, 3.0], [3.0, 2.0, 1.0]])
    expected = torch.log_softmax(torch.tensor(batch), dim=-1).numpy()
    assert log_softmax(batch) == pytest.approx(expected)


def test_log_softmax_honours_temperature(logits):
    log_probs = log_softmax(logits, temperature=10.0)
    assert np.exp(log_probs) == pytest.approx(softmax_temperature(logits, temperature=10.0))


def test_log_softmax_stays_finite_on_extreme_spread():
    # log(softmax(x)) underflows to -inf here; the stable form
    # shifted - log(sum(exp(shifted))) stays finite.
    assert np.isfinite(log_softmax(np.array([1e9, -1e9, 0.0]))).all()


............                                                                                 [100%]


### A2. Cross-entropy (binary + multiclass)
**Watch for:** clipping / log-softmax; mean vs sum.
**Follow-ups:** label smoothing; class weights; fused CE+softmax.


In [78]:
def binary_cross_entropy(y_true, y_prob, eps=1e-7):
    """y_true in {0,1}, y_prob in (0,1). Return mean BCE."""

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    bce = y_true * np.log(y_prob + eps) + (1 - y_true) * np.log(1 - y_prob + eps) 

    return - bce.mean()


def cross_entropy(y_true, logits):
    """y_true: (N,) int labels; logits: (N, C). Return mean CE."""
    
    logits = np.array(logits)
    rows = [i for i in range(len(y_true))]

    shifted = logits - logits.max(axis=-1, keepdims=True)
    exp = np.exp(shifted)
    ce = - (np.log(exp) - np.log(exp.sum(axis=-1, keepdims=True)))[rows, y_true]
    return ce.mean()


In [79]:
y_true = [0, 1, 2, 3]
logits = np.zeros((4,5))

rows = [i for i in range(len(y_true))]
ce = - np.log(logits[rows, y_true] + 1e-6)

ce, logits

(array([13.81551056, 13.81551056, 13.81551056, 13.81551056]),
 array([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]]))

In [80]:
%%ipytest -qq
# Unit tests — A2 cross-entropy


def test_bce_is_small_when_confident_and_correct():
    assert binary_cross_entropy([1, 0], [0.9, 0.1]) < 0.2


def test_bce_penalises_confident_mistakes():
    confident_wrong = binary_cross_entropy([1, 0], [0.1, 0.9])
    confident_right = binary_cross_entropy([1, 0], [0.9, 0.1])
    assert confident_wrong > confident_right


def test_bce_clips_extreme_probabilities():
    # p exactly 0 or 1 must not produce inf/nan
    assert np.isfinite(binary_cross_entropy([1, 0], [1.0, 0.0]))
    assert np.isfinite(binary_cross_entropy([1, 0], [0.0, 1.0]))


def test_bce_matches_torch():
    y_true = [1.0, 0.0, 1.0]
    y_prob = [0.8, 0.3, 0.6]
    expected = torch.nn.functional.binary_cross_entropy(
        torch.tensor(y_prob), torch.tensor(y_true)
    ).item()
    assert binary_cross_entropy(y_true, y_prob) == pytest.approx(expected, abs=1e-6)


def test_cross_entropy_is_small_when_logits_favour_true_class():
    logits = np.array([[2.0, 0.0], [0.0, 2.0]])
    assert cross_entropy([0, 1], logits) < 0.2


def test_cross_entropy_of_uniform_logits_is_log_num_classes():
    logits = np.zeros((4, 5))
    assert cross_entropy([0, 1, 2, 3], logits) == pytest.approx(math.log(5))


def test_cross_entropy_matches_torch():
    logits = np.array([[2.0, 0.5, -1.0], [0.1, 0.2, 3.0]])
    labels = [0, 2]
    expected = torch.nn.functional.cross_entropy(
        torch.tensor(logits), torch.tensor(labels)
    ).item()
    assert cross_entropy(labels, logits) == pytest.approx(expected, abs=1e-6)


def test_cross_entropy_is_stable_for_large_logits():
    logits = np.array([[1e4, 0.0], [0.0, 1e4]])
    assert np.isfinite(cross_entropy([0, 1], logits))


........                                                                                     [100%]
========================================= warnings summary =========================================
t_cc6dc959bf2f4b388b43cdf9e45f1aa0.py::test_cross_entropy_is_stable_for_large_logits
  /var/folders/z5/pmqbnwgn69v41qrwhq_t5tc80000gn/T/ipykernel_21721/2523012413.py:19: RuntimeWarning: divide by zero encountered in log
    ce = - (np.log(exp) - np.log(exp.sum(axis=-1, keepdims=True)))[rows, y_true]

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html


### A3. Confusion-matrix primitives
**Watch for:** O(N) indexing; label range; empty input.
**Follow-ups:** macro-F1 from the matrix.


In [ ]:
def confusion_counts(y_true, y_pred, n_classes):
    """Return C where C[i, j] = count of true=i, pred=j. No sklearn."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — A3 confusion matrix


@pytest.fixture
def counts():
    return confusion_counts([0, 1, 1, 0], [0, 1, 0, 0], n_classes=2)


def test_shape_matches_num_classes(counts):
    assert counts.shape == (2, 2)


def test_entries_count_true_predicted_pairs(counts):
    assert counts[0, 0] == 2  # true 0, predicted 0
    assert counts[1, 1] == 1  # true 1, predicted 1
    assert counts[1, 0] == 1  # true 1, predicted 0
    assert counts[0, 1] == 0


def test_total_equals_number_of_samples(counts):
    assert counts.sum() == 4


def test_perfect_predictions_are_diagonal():
    labels = [0, 1, 2, 2]
    counts = confusion_counts(labels, labels, n_classes=3)
    assert np.array_equal(counts, np.diag([1, 1, 2]))


def test_empty_input_gives_zero_matrix():
    counts = confusion_counts([], [], n_classes=3)
    assert counts.shape == (3, 3)
    assert counts.sum() == 0


def test_recall_can_be_derived_from_matrix(counts):
    recall_class_1 = counts[1, 1] / counts[1].sum()
    assert recall_class_1 == pytest.approx(0.5)


### A4. Top-k accuracy
**Watch for:** argpartition/argsort; k > C; ties.


In [ ]:
def top_k_accuracy(logits, y_true, k=5):
    """logits (N,C), y_true (N,). Fraction where true label is in top-k."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — A4 top-k accuracy

LOGITS = np.array(
    [
        [0.1, 0.2, 0.9, 0.0],
        [0.8, 0.1, 0.05, 0.05],
    ]
)


@pytest.mark.parametrize(
    "y_true, k, expected",
    [
        ([2, 0], 1, 1.0),  # both argmax predictions correct
        ([1, 2], 1, 0.0),  # neither is argmax
        ([1, 2], 2, 0.5),  # only the first is inside its top-2
        ([1, 1], 2, 1.0),  # both labels inside their top-2
    ],
)
def test_known_values(y_true, k, expected):
    assert top_k_accuracy(LOGITS, y_true, k=k) == pytest.approx(expected)


def test_k_equal_to_num_classes_is_always_perfect():
    assert top_k_accuracy(LOGITS, [3, 3], k=LOGITS.shape[1]) == pytest.approx(1.0)


def test_k_larger_than_num_classes_does_not_crash():
    assert top_k_accuracy(LOGITS, [1, 2], k=99) == pytest.approx(1.0)


def test_accuracy_is_monotonic_in_k():
    scores = [top_k_accuracy(LOGITS, [1, 2], k=k) for k in (1, 2, 3, 4)]
    assert scores == sorted(scores)


def test_top_1_matches_argmax_accuracy():
    y_true = np.array([2, 1])
    expected = (LOGITS.argmax(axis=1) == y_true).mean()
    assert top_k_accuracy(LOGITS, y_true, k=1) == pytest.approx(expected)


### A5. IoU / Dice
**Watch for:** intersection clamp; union; divide-by-zero.
**Physical AI:** NMS uses IoU.


In [ ]:
def binary_iou(mask_true, mask_pred):
    """Boolean or 0/1 arrays, same shape. Return IoU in [0,1]."""
    # YOUR CODE HERE
    raise NotImplementedError


def box_iou(box_a, box_b):
    """Each box = (x1, y1, x2, y2). Handle no-overlap and zero-area."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — A5 IoU


def test_binary_iou_known_value():
    assert binary_iou([1, 1, 0, 0], [1, 0, 1, 0]) == pytest.approx(1 / 3)


def test_binary_iou_identical_masks_is_one():
    mask = [1, 0, 1, 1]
    assert binary_iou(mask, mask) == pytest.approx(1.0)


def test_binary_iou_disjoint_masks_is_zero():
    assert binary_iou([1, 1, 0, 0], [0, 0, 1, 1]) == pytest.approx(0.0)


def test_binary_iou_empty_masks_avoids_zero_division():
    assert binary_iou([0, 0], [0, 0]) == pytest.approx(0.0)


@pytest.mark.parametrize(
    "box_a, box_b, expected",
    [
        ((0, 0, 2, 2), (1, 1, 3, 3), 1 / 7),  # partial overlap
        ((0, 0, 1, 1), (2, 2, 3, 3), 0.0),  # no overlap
        ((0, 0, 2, 2), (0, 0, 2, 2), 1.0),  # identical
        ((0, 0, 4, 4), (1, 1, 2, 2), 1 / 16),  # fully contained
    ],
)
def test_box_iou_known_values(box_a, box_b, expected):
    assert box_iou(box_a, box_b) == pytest.approx(expected)


def test_box_iou_is_symmetric():
    box_a, box_b = (0, 0, 2, 2), (1, 1, 3, 3)
    assert box_iou(box_a, box_b) == pytest.approx(box_iou(box_b, box_a))


def test_box_iou_zero_area_box_avoids_zero_division():
    assert box_iou((0, 0, 0, 0), (0, 0, 0, 0)) == pytest.approx(0.0)


def test_boxes_touching_at_edge_have_zero_iou():
    assert box_iou((0, 0, 1, 1), (1, 0, 2, 1)) == pytest.approx(0.0)


### A6. Exponential moving average
**Follow-ups:** BN running stats; calibration observers; QAT.


In [ ]:
def exponential_moving_average(values, momentum=0.9):
    """Return EMA state after each update: ema = momentum * ema + (1 - momentum) * x"""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — A6 exponential moving average


def test_first_value_initialises_the_state():
    assert exponential_moving_average([10.0], momentum=0.9)[0] == pytest.approx(10.0)


def test_update_rule():
    ema = exponential_moving_average([10.0, 0.0], momentum=0.9)
    assert ema[1] == pytest.approx(0.9 * 10.0 + 0.1 * 0.0)


def test_output_length_matches_input():
    values = [1.0, 2.0, 3.0, 4.0]
    assert len(exponential_moving_average(values)) == len(values)


def test_constant_stream_stays_constant():
    ema = exponential_moving_average([5.0] * 5, momentum=0.9)
    assert np.asarray(ema) == pytest.approx(np.full(5, 5.0))


@pytest.mark.parametrize("momentum", [0.1, 0.5, 0.99])
def test_state_stays_between_previous_state_and_new_value(momentum):
    ema = exponential_moving_average([0.0, 1.0], momentum=momentum)
    assert 0.0 <= ema[1] <= 1.0


def test_lower_momentum_tracks_new_values_faster():
    fast = exponential_moving_average([0.0, 1.0], momentum=0.1)[1]
    slow = exponential_moving_average([0.0, 1.0], momentum=0.9)[1]
    assert fast > slow


### A7. Affine quantize / dequantize
**Watch for:** rounding; clip; per-channel broadcast.
**Follow-ups:** per-tensor vs per-channel; symmetric vs asymmetric.


In [ ]:
def quantize_affine(x, scale, zero_point, qmin=-128, qmax=127):
    """q = clip(round(x / scale) + zero_point, qmin, qmax)"""
    # YOUR CODE HERE
    raise NotImplementedError


def dequantize_affine(q, scale, zero_point):
    """x_hat = scale * (q - zero_point)"""
    # YOUR CODE HERE
    raise NotImplementedError


def sqnr_db(x, x_hat, eps=1e-12):
    signal = np.mean(x ** 2)
    noise = np.mean((x - x_hat) ** 2)
    return 10 * np.log10(signal / (noise + eps))


In [ ]:
%%ipytest -qq
# Unit tests — A7 affine quantize / dequantize

SCALE = 1 / 127


@pytest.fixture
def signal():
    return np.linspace(-1, 1, 100)


def test_quantize_returns_integers(signal):
    q = quantize_affine(signal, scale=SCALE, zero_point=0, qmin=-127, qmax=127)
    assert np.issubdtype(np.asarray(q).dtype, np.integer)


def test_quantize_clips_to_representable_range():
    q = quantize_affine(np.array([-10.0, 10.0]), scale=SCALE, zero_point=0, qmin=-127, qmax=127)
    assert q.min() == -127
    assert q.max() == 127


def test_round_trip_keeps_sqnr_above_30db(signal):
    q = quantize_affine(signal, scale=SCALE, zero_point=0, qmin=-127, qmax=127)
    reconstructed = dequantize_affine(q, scale=SCALE, zero_point=0)
    assert sqnr_db(signal, reconstructed) > 30


def test_round_trip_error_bounded_by_half_a_step(signal):
    q = quantize_affine(signal, scale=SCALE, zero_point=0, qmin=-127, qmax=127)
    reconstructed = dequantize_affine(q, scale=SCALE, zero_point=0)
    assert np.abs(reconstructed - signal).max() <= SCALE / 2 + 1e-9


def test_zero_point_maps_zero_to_itself():
    q = quantize_affine(np.array([0.0]), scale=SCALE, zero_point=7, qmin=-127, qmax=127)
    assert q[0] == 7
    assert dequantize_affine(q, scale=SCALE, zero_point=7)[0] == pytest.approx(0.0)


def test_dequantize_is_exact_on_grid_points():
    q = np.array([-5, 0, 5])
    expected = SCALE * (q - 2)
    assert dequantize_affine(q, scale=SCALE, zero_point=2) == pytest.approx(expected)


def test_sqnr_is_higher_for_finer_scale(signal):
    coarse = dequantize_affine(
        quantize_affine(signal, scale=1 / 7, zero_point=0, qmin=-7, qmax=7), 1 / 7, 0
    )
    fine = dequantize_affine(
        quantize_affine(signal, scale=SCALE, zero_point=0, qmin=-127, qmax=127), SCALE, 0
    )
    assert sqnr_db(signal, fine) > sqnr_db(signal, coarse)


### A8. Calibration params from range
Formulas:
- `scale = (x_max - x_min) / (qmax - qmin)`
- `zero_point = clip(round(qmin - x_min / scale), qmin, qmax)`
Handle `x_min == x_max`.


In [ ]:
def calibration_params(x_min, x_max, qmin=0, qmax=255):
    """Return (scale, zero_point) for affine quantization."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — A8 calibration parameters


def test_unit_range_uint8():
    scale, zero_point = calibration_params(0.0, 1.0, qmin=0, qmax=255)
    assert scale == pytest.approx(1 / 255)
    assert zero_point == 0


def test_symmetric_range_puts_zero_point_mid_scale():
    scale, zero_point = calibration_params(-1.0, 1.0, qmin=0, qmax=255)
    assert scale == pytest.approx(2 / 255)
    assert zero_point == pytest.approx(128, abs=1)


def test_degenerate_range_still_returns_positive_scale():
    scale, _ = calibration_params(5.0, 5.0)
    assert scale > 0


def test_zero_point_stays_inside_integer_range():
    for x_min, x_max in [(-10.0, -1.0), (1.0, 10.0), (-3.0, 7.0)]:
        _, zero_point = calibration_params(x_min, x_max, qmin=0, qmax=255)
        assert 0 <= zero_point <= 255


def test_round_trip_error_within_one_step():
    scale, zero_point = calibration_params(-1.0, 1.0, qmin=0, qmax=255)
    x = np.array([-1.0, -0.5, 0.0, 0.25, 1.0])
    reconstructed = dequantize_affine(
        quantize_affine(x, scale, zero_point, qmin=0, qmax=255), scale, zero_point
    )
    assert np.abs(reconstructed - x).max() <= scale


def test_wider_range_gives_coarser_scale():
    narrow, _ = calibration_params(-1.0, 1.0)
    wide, _ = calibration_params(-100.0, 100.0)
    assert wide > narrow


---
# Tier B — PyTorch fluency


### B1. Tiny `nn.Module`
Linear → ReLU → Linear → logits (no softmax if using CE).


In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, in_dim, hidden, n_classes):
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError

    def forward(self, x):
        # YOUR CODE HERE
        raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — B1 MLPClassifier


@pytest.fixture
def model():
    return MLPClassifier(4, 8, 3)


def test_output_shape(model):
    assert model(torch.randn(2, 4)).shape == (2, 3)


def test_has_two_linear_layers(model):
    linears = [m for m in model.modules() if isinstance(m, nn.Linear)]
    assert len(linears) == 2
    assert linears[0].in_features == 4
    assert linears[-1].out_features == 3


def test_returns_logits_not_probabilities(model):
    # CE expects logits, so no Softmax inside forward
    assert not any(isinstance(m, nn.Softmax) for m in model.modules())


def test_forward_is_differentiable(model):
    loss = model(torch.randn(2, 4)).sum()
    loss.backward()
    assert all(p.grad is not None for p in model.parameters())


def test_handles_single_sample_batch(model):
    assert model(torch.randn(1, 4)).shape == (1, 3)


### B2. Manual SGD step
Include L2: `grad += weight_decay * param`.


In [ ]:
def sgd_step(params, grads, lr, weight_decay=0.0):
    """Return updated params (list of tensors). Do not use optimizer.step."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — B2 manual SGD step


def test_basic_update():
    updated = sgd_step([torch.tensor([1.0, 2.0])], [torch.tensor([0.5, -0.5])], lr=0.1)
    assert torch.allclose(updated[0], torch.tensor([0.95, 2.05]))


def test_zero_learning_rate_is_a_noop():
    param = torch.tensor([1.0, -3.0])
    updated = sgd_step([param], [torch.tensor([9.0, 9.0])], lr=0.0)
    assert torch.allclose(updated[0], param)


def test_weight_decay_pulls_parameters_towards_zero():
    param = torch.tensor([2.0])
    no_decay = sgd_step([param], [torch.zeros(1)], lr=0.1, weight_decay=0.0)[0]
    with_decay = sgd_step([param], [torch.zeros(1)], lr=0.1, weight_decay=0.5)[0]
    assert with_decay.item() < no_decay.item() == pytest.approx(2.0)


def test_weight_decay_matches_l2_gradient():
    param = torch.tensor([2.0])
    grad = torch.tensor([1.0])
    expected = param - 0.1 * (grad + 0.5 * param)
    updated = sgd_step([param], [grad], lr=0.1, weight_decay=0.5)[0]
    assert torch.allclose(updated, expected)


def test_updates_every_parameter():
    params = [torch.tensor([1.0]), torch.tensor([[1.0, 1.0]])]
    grads = [torch.tensor([1.0]), torch.ones(1, 2)]
    updated = sgd_step(params, grads, lr=0.5)
    assert len(updated) == 2
    assert torch.allclose(updated[0], torch.tensor([0.5]))
    assert torch.allclose(updated[1], torch.full((1, 2), 0.5))


### B3. Sliding-window Dataset
`__len__` and `__getitem__` must be consistent (off-by-one traps).


In [ ]:
class SlidingWindowDataset(Dataset):
    def __init__(self, signal, window):
        # YOUR CODE HERE
        raise NotImplementedError

    def __len__(self):
        # YOUR CODE HERE
        raise NotImplementedError

    def __getitem__(self, idx):
        # YOUR CODE HERE
        raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — B3 sliding-window dataset


@pytest.fixture
def dataset():
    return SlidingWindowDataset(np.arange(5), window=3)


def test_length_counts_all_windows(dataset):
    assert len(dataset) == 3


@pytest.mark.parametrize(
    "idx, expected",
    [(0, [0, 1, 2]), (1, [1, 2, 3]), (2, [2, 3, 4])],
)
def test_items_are_consecutive_windows(dataset, idx, expected):
    assert torch.as_tensor(dataset[idx]).tolist() == expected


def test_every_index_in_range_is_reachable(dataset):
    assert all(torch.as_tensor(dataset[i]).shape == (3,) for i in range(len(dataset)))


def test_window_equal_to_signal_length_yields_one_item():
    assert len(SlidingWindowDataset(np.arange(3), window=3)) == 1


def test_window_longer_than_signal_is_empty():
    assert len(SlidingWindowDataset(np.arange(2), window=3)) == 0


def test_works_with_dataloader(dataset):
    batch = next(iter(DataLoader(dataset, batch_size=2)))
    assert batch.shape == (2, 3)


### B4. Causal / padding masks
**Follow-ups:** FlashAttention; why materializing N×N hurts.


In [ ]:
def causal_mask(seq_len):
    """Boolean mask where position i cannot attend to j > i.
    Shape (seq_len, seq_len). True means MASKED (blocked).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def key_padding_mask(lengths, max_len):
    """True for pad positions. lengths: list/array of ints. Shape (B, max_len)."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — B4 attention masks


@pytest.fixture
def mask():
    return np.asarray(causal_mask(3))


def test_causal_mask_shape_and_dtype(mask):
    assert mask.shape == (3, 3)
    assert mask.dtype == bool


def test_causal_mask_allows_self_and_past(mask):
    assert not mask[0, 0]
    assert not mask[2, 0]


def test_causal_mask_blocks_future(mask):
    assert mask[0, 2]
    assert mask[1, 2]


def test_causal_mask_matches_upper_triangle(mask):
    assert np.array_equal(mask, np.triu(np.ones((3, 3), dtype=bool), k=1))


def test_key_padding_mask_marks_pad_positions():
    padding = np.asarray(key_padding_mask([2, 3], max_len=3))
    assert padding.shape == (2, 3)
    assert padding[0].tolist() == [False, False, True]
    assert padding[1].tolist() == [False, False, False]


def test_key_padding_mask_counts_match_lengths():
    lengths = [1, 2, 4]
    padding = np.asarray(key_padding_mask(lengths, max_len=4))
    assert (~padding).sum(axis=-1).tolist() == lengths


### B5. Layer-wise activation dump (hooks)
Clean up hooks afterward. Useful for ONNX / PTQ debugging.


In [ ]:
def collect_activations(model, x, layer_names):
    """Run one forward; return dict name -> tensor (detached cpu)."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — B5 activation collection via hooks


class Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 4)
        self.fc2 = nn.Linear(4, 2)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))


@pytest.fixture
def model():
    return Tiny()


@pytest.fixture
def activations(model):
    return collect_activations(model, torch.randn(3, 4), ["fc1", "fc2"])


def test_returns_only_requested_layers(activations):
    assert set(activations) == {"fc1", "fc2"}


def test_shapes_match_layer_outputs(activations):
    assert activations["fc1"].shape == (3, 4)
    assert activations["fc2"].shape == (3, 2)


def test_can_collect_a_single_layer(model):
    activations = collect_activations(model, torch.randn(1, 4), ["fc2"])
    assert list(activations) == ["fc2"]


def test_tensors_are_detached(activations):
    assert not any(tensor.requires_grad for tensor in activations.values())


def test_hooks_are_removed_afterwards(model, activations):
    assert len(model.fc1._forward_hooks) == 0
    assert len(model.fc2._forward_hooks) == 0


### B6. MinMax PTQ observer


In [ ]:
class MinMaxObserver:
    def __init__(self):
        self.min_val = None
        self.max_val = None

    def update(self, x):
        """Update running min/max over calibration batches."""
        # YOUR CODE HERE
        raise NotImplementedError

    def compute_qparams(self, qmin, qmax):
        """Return (scale, zero_point)."""
        # YOUR CODE HERE
        raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — B6 MinMax observer


@pytest.fixture
def observer():
    obs = MinMaxObserver()
    obs.update(torch.tensor([-1.0, 0.5]))
    obs.update(torch.tensor([2.0]))
    return obs


def test_tracks_running_min_and_max(observer):
    assert float(observer.min_val) == pytest.approx(-1.0)
    assert float(observer.max_val) == pytest.approx(2.0)


def test_narrower_batch_does_not_shrink_the_range(observer):
    observer.update(torch.tensor([0.0]))
    assert float(observer.min_val) == pytest.approx(-1.0)
    assert float(observer.max_val) == pytest.approx(2.0)


def test_qparams_are_valid(observer):
    scale, zero_point = observer.compute_qparams(0, 255)
    assert scale > 0
    assert 0 <= zero_point <= 255


def test_is_order_invariant():
    forward, backward = MinMaxObserver(), MinMaxObserver()
    batches = [torch.tensor([-1.0, 0.5]), torch.tensor([2.0])]
    for batch in batches:
        forward.update(batch)
    for batch in reversed(batches):
        backward.update(batch)
    assert forward.compute_qparams(0, 255) == pytest.approx(backward.compute_qparams(0, 255))


def test_calibrated_range_covers_observed_values(observer):
    scale, zero_point = observer.compute_qparams(0, 255)
    values = np.array([-1.0, 0.0, 2.0])
    reconstructed = dequantize_affine(
        quantize_affine(values, scale, zero_point, qmin=0, qmax=255), scale, zero_point
    )
    assert np.abs(reconstructed - values).max() <= scale


---
# Tier C — NumPy / performance


### C1. Vectorized pairwise L2
`a: (N,D), b: (M,D)` → distances `(N,M)`. Avoid naive Python loops.


In [ ]:
def pairwise_l2(a, b):
    """Return (N, M) pairwise Euclidean distances."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — C1 pairwise L2


@pytest.fixture
def points():
    a = np.array([[0.0, 0.0], [1.0, 0.0]])
    b = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
    return a, b


def test_shape_is_n_by_m(points):
    a, b = points
    assert pairwise_l2(a, b).shape == (len(a), len(b))


def test_known_distances(points):
    a, b = points
    distances = pairwise_l2(a, b)
    assert distances[0, 0] == pytest.approx(0.0)
    assert distances[0, 1] == pytest.approx(1.0)
    assert distances[1, 2] == pytest.approx(1.0)
    assert distances[0, 2] == pytest.approx(math.sqrt(2))


def test_matches_naive_double_loop():
    rng = np.random.default_rng(0)
    a, b = rng.normal(size=(5, 3)), rng.normal(size=(4, 3))
    expected = np.array([[np.linalg.norm(p - q) for q in b] for p in a])
    assert pairwise_l2(a, b) == pytest.approx(expected)


def test_self_distances_are_zero_and_never_negative():
    rng = np.random.default_rng(1)
    a = rng.normal(size=(6, 4))
    distances = pairwise_l2(a, a)
    assert np.diag(distances) == pytest.approx(np.zeros(6), abs=1e-6)
    assert (distances >= 0).all()


def test_is_symmetric_for_identical_inputs():
    rng = np.random.default_rng(2)
    a = rng.normal(size=(4, 2))
    distances = pairwise_l2(a, a)
    assert distances == pytest.approx(distances.T)


### C2. One-hot + batched gather


In [ ]:
def one_hot(indices, n_classes):
    """indices (N,) -> (N, C) float array."""
    # YOUR CODE HERE
    raise NotImplementedError


def gather_rows(mat, indices):
    """mat (N, C), indices (N,) -> (N,) values mat[i, indices[i]]."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — C2 one-hot and gather


def test_one_hot_shape_and_rows():
    encoded = one_hot([0, 2], 3)
    assert encoded.shape == (2, 3)
    assert encoded == pytest.approx(np.array([[1.0, 0.0, 0.0], [0.0, 0.0, 1.0]]))


def test_one_hot_rows_sum_to_one():
    encoded = one_hot([0, 1, 2, 2], 3)
    assert encoded.sum(axis=1) == pytest.approx(np.ones(4))


def test_one_hot_argmax_recovers_indices():
    indices = [2, 0, 1]
    assert one_hot(indices, 3).argmax(axis=1).tolist() == indices


def test_gather_rows_picks_expected_values():
    matrix = np.array([[1.0, 2.0], [3.0, 4.0]])
    assert gather_rows(matrix, [1, 0]) == pytest.approx(np.array([2.0, 3.0]))


def test_gather_rows_output_is_one_dimensional():
    matrix = np.arange(12, dtype=float).reshape(4, 3)
    assert gather_rows(matrix, [0, 1, 2, 0]).shape == (4,)


def test_gather_rows_of_one_hot_returns_ones():
    encoded = one_hot([0, 2], 3)
    assert gather_rows(encoded, [0, 2]) == pytest.approx(np.ones(2))


### C3. NMS (non-maximum suppression)
Physical AI / perception flavored.


In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    """boxes (N,4) xyxy, scores (N,). Return kept indices in score order."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — C3 non-maximum suppression

BOXES = np.array(
    [
        [0.0, 0.0, 2.0, 2.0],
        [0.5, 0.5, 2.5, 2.5],  # heavily overlaps box 0
        [5.0, 5.0, 7.0, 7.0],  # isolated
    ]
)
SCORES = np.array([0.9, 0.8, 0.7])


def test_suppresses_overlapping_lower_scoring_box():
    assert list(nms(BOXES, SCORES, iou_threshold=0.3)) == [0, 2]


def test_high_threshold_keeps_everything():
    assert sorted(nms(BOXES, SCORES, iou_threshold=0.99)) == [0, 1, 2]


def test_returns_indices_in_descending_score_order():
    scores = np.array([0.1, 0.5, 0.9])
    kept = list(nms(BOXES, scores, iou_threshold=0.99))
    assert kept == sorted(kept, key=lambda i: -scores[i])


def test_single_box_is_always_kept():
    assert list(nms(BOXES[:1], SCORES[:1], iou_threshold=0.5)) == [0]


def test_identical_boxes_collapse_to_one():
    duplicated = np.repeat(BOXES[:1], 3, axis=0)
    assert len(nms(duplicated, np.array([0.9, 0.8, 0.7]), iou_threshold=0.5)) == 1


def test_kept_boxes_do_not_overlap_above_threshold():
    threshold = 0.3
    kept = list(nms(BOXES, SCORES, iou_threshold=threshold))
    for i, first in enumerate(kept):
        for second in kept[i + 1 :]:
            assert box_iou(BOXES[first], BOXES[second]) <= threshold


### C4. Streaming mean/std (Welford)


In [ ]:
def streaming_mean_std(stream):
    """One-pass mean and sample std (ddof=1 if n>1 else 0)."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — C4 streaming mean and std


def test_matches_numpy_on_a_small_sample():
    values = [1.0, 2.0, 3.0]
    mean, std = streaming_mean_std(values)
    assert mean == pytest.approx(np.mean(values))
    assert std == pytest.approx(np.std(values, ddof=1))


def test_matches_numpy_on_random_data():
    values = np.random.default_rng(0).normal(size=1000).tolist()
    mean, std = streaming_mean_std(values)
    assert mean == pytest.approx(np.mean(values))
    assert std == pytest.approx(np.std(values, ddof=1))


def test_single_value_has_zero_std():
    mean, std = streaming_mean_std([7.0])
    assert mean == pytest.approx(7.0)
    assert std == pytest.approx(0.0)


def test_constant_stream_has_zero_std():
    _, std = streaming_mean_std([4.0] * 10)
    assert std == pytest.approx(0.0)


def test_consumes_a_generator_in_one_pass():
    mean, _ = streaming_mean_std(float(x) for x in range(5))
    assert mean == pytest.approx(2.0)


def test_is_numerically_stable_with_large_offset():
    values = [1e9 + x for x in (1.0, 2.0, 3.0)]
    _, std = streaming_mean_std(values)
    assert std == pytest.approx(1.0, abs=1e-3)


### C5. Concurrent JPEG decode
Threads are fine when the native decoder releases the GIL.


In [ ]:
def decode_jpeg(jpeg_bytes: bytes):
    import cv2
    buf = np.frombuffer(jpeg_bytes, dtype=np.uint8)
    img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("failed to decode")
    return img


def decode_many(jpeg_bytes_list, max_workers=4):
    """Decode list of JPEG bytes -> list of HxWxC arrays."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — C5 concurrent JPEG decode
import cv2


@pytest.fixture
def jpeg_blob():
    ok, buffer = cv2.imencode(".jpg", np.zeros((8, 8, 3), dtype=np.uint8))
    assert ok, "fixture setup: encoding failed"
    return buffer.tobytes()


def test_decodes_every_blob(jpeg_blob):
    images = decode_many([jpeg_blob] * 3, max_workers=2)
    assert len(images) == 3
    assert all(image.shape == (8, 8, 3) for image in images)


def test_preserves_input_order(jpeg_blob):
    small = jpeg_blob
    ok, buffer = cv2.imencode(".jpg", np.zeros((4, 4, 3), dtype=np.uint8))
    assert ok
    large = buffer.tobytes()
    images = decode_many([small, large], max_workers=2)
    assert [image.shape[0] for image in images] == [8, 4]


def test_matches_sequential_decode(jpeg_blob):
    expected = decode_jpeg(jpeg_blob)
    assert np.array_equal(decode_many([jpeg_blob])[0], expected)


def test_empty_input_returns_empty_list():
    assert decode_many([], max_workers=2) == []


def test_invalid_bytes_raise(jpeg_blob):
    with pytest.raises(Exception):
        decode_many([b"not a jpeg"], max_workers=1)


---
# Tier D — Debugging exercises
Fix the broken snippets. Do not rewrite from scratch unless necessary.


### D1. Broken F1


In [ ]:
def precision_recall_f1_broken(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    # BUGS: `and` instead of `&`; no zero-division guard; ~ on ints
    tp = ((y_true == 1) and (y_pred == 1)).sum()
    fp = ((~y_true) and (y_pred == 1)).sum()
    fn = ((y_true == 1) and (y_pred == 0)).sum()
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1


def precision_recall_f1_fixed(y_true, y_pred):
    # YOUR FIX HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — D1 broken vs fixed F1


def test_broken_version_raises_on_array_truthiness():
    with pytest.raises(ValueError):
        precision_recall_f1_broken([1, 0, 1], [1, 1, 0])


def test_fixed_version_known_values():
    precision, recall, _ = precision_recall_f1_fixed([1, 0, 1], [1, 1, 0])
    assert precision == pytest.approx(0.5)
    assert recall == pytest.approx(0.5)


def test_fixed_version_guards_zero_division():
    assert precision_recall_f1_fixed([0, 0], [0, 0]) == (0.0, 0.0, 0.0)


def test_fixed_version_handles_all_positive_labels():
    precision, recall, f1 = precision_recall_f1_fixed([1, 1], [1, 1])
    assert (precision, recall, f1) == pytest.approx((1.0, 1.0, 1.0))


def test_fixed_version_agrees_with_warmup_implementation():
    y_true, y_pred = [1, 1, 0, 0, 1], [1, 0, 0, 1, 1]
    assert precision_recall_f1_fixed(y_true, y_pred) == pytest.approx(
        precision_recall_f1(y_true, y_pred)
    )


### D2. Softmax overflow


In [ ]:
def softmax_broken(x):
    e = np.exp(x)
    return e / e.sum()


def softmax_fixed(x):
    # YOUR FIX HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — D2 softmax overflow


LARGE = np.array([1000.0, 1001.0, 1002.0])


def test_broken_version_overflows():
    with np.errstate(over="ignore", invalid="ignore"):
        assert not np.isfinite(softmax_broken(LARGE)).all()


def test_fixed_version_is_finite_and_normalised():
    probs = softmax_fixed(LARGE)
    assert np.isfinite(probs).all()
    assert probs.sum() == pytest.approx(1.0)


def test_fixed_version_preserves_argmax():
    assert np.argmax(softmax_fixed(LARGE)) == 2


def test_fixed_version_matches_torch_on_moderate_input():
    moderate = np.array([1.0, 2.0, 3.0])
    expected = torch.softmax(torch.tensor(moderate), dim=-1).numpy()
    assert softmax_fixed(moderate) == pytest.approx(expected)


def test_both_agree_when_no_overflow():
    moderate = np.array([0.5, -0.5, 1.5])
    assert softmax_fixed(moderate) == pytest.approx(softmax_broken(moderate))


### D3. Train/eval mismatch
Explain and fix: Dropout/BN left in train mode at inference.


In [ ]:
class DropNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)
        self.drop = nn.Dropout(0.9)

    def forward(self, x):
        return self.drop(self.fc(x))


def predict_broken(model, x):
    # BUG: model left in train mode
    return model(x)


def predict_fixed(model, x):
    # YOUR FIX HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — D3 train/eval mismatch


@pytest.fixture
def model():
    torch.manual_seed(0)
    return DropNet()


@pytest.fixture
def batch():
    torch.manual_seed(1)
    return torch.randn(32, 4)


def test_broken_version_is_nondeterministic(model, batch):
    with torch.no_grad():
        first, second = predict_broken(model, batch), predict_broken(model, batch)
    assert not torch.allclose(first, second)


def test_fixed_version_is_deterministic(model, batch):
    with torch.no_grad():
        first, second = predict_fixed(model, batch), predict_fixed(model, batch)
    assert torch.allclose(first, second)


def test_fixed_version_switches_module_to_eval(model, batch):
    with torch.no_grad():
        predict_fixed(model, batch)
    assert not model.training


def test_fixed_version_disables_dropout_scaling(model, batch):
    # in eval mode dropout is identity, so output == linear layer output
    with torch.no_grad():
        assert torch.allclose(predict_fixed(model, batch), model.fc(batch))


def test_output_shape_is_unchanged(model, batch):
    with torch.no_grad():
        assert predict_fixed(model, batch).shape == (32, 2)


### D4. In-place autograd bug


In [ ]:
def broken_inplace(x):
    # x is a leaf that requires grad
    x += 1
    return x.sum()


def fixed_no_inplace(x):
    # YOUR FIX HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — D4 in-place autograd bug


def test_broken_version_raises_on_leaf_mutation():
    x = torch.tensor([1.0, 2.0], requires_grad=True)
    with pytest.raises(RuntimeError):
        broken_inplace(x)


def test_fixed_version_produces_gradients():
    x = torch.tensor([1.0, 2.0], requires_grad=True)
    fixed_no_inplace(x).backward()
    assert x.grad is not None
    assert torch.allclose(x.grad, torch.ones(2))


def test_fixed_version_does_not_mutate_input():
    x = torch.tensor([1.0, 2.0], requires_grad=True)
    fixed_no_inplace(x)
    assert torch.allclose(x.detach(), torch.tensor([1.0, 2.0]))


def test_fixed_version_returns_expected_value():
    x = torch.tensor([1.0, 2.0], requires_grad=True)
    assert fixed_no_inplace(x).item() == pytest.approx(5.0)


def test_fixed_version_works_without_grad():
    x = torch.tensor([1.0, 2.0])
    assert fixed_no_inplace(x).item() == pytest.approx(5.0)


### D5. Attention score shape / scaling bug


In [ ]:
def attention_scores_broken(Q, K):
    # BUGS: wrong transpose; missing 1/sqrt(d); mask broadcast
    return Q @ K


def attention_scores_fixed(Q, K, mask=None):
    """Q,K: (B, H, T, D). Return scores (B, H, T, T), scaled.
    Optional mask: True means masked (set to -inf).
    """
    # YOUR FIX HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — D5 attention scores

B, H, T, D = 2, 3, 4, 8


@pytest.fixture
def qk():
    torch.manual_seed(0)
    return torch.randn(B, H, T, D), torch.randn(B, H, T, D)


def test_broken_version_has_shape_mismatch(qk):
    Q, K = qk
    with pytest.raises(RuntimeError):
        attention_scores_broken(Q, K)


def test_scores_shape(qk):
    Q, K = qk
    assert attention_scores_fixed(Q, K).shape == (B, H, T, T)


def test_scores_are_scaled_by_sqrt_of_head_dim(qk):
    Q, K = qk
    expected = Q @ K.transpose(-2, -1) / math.sqrt(D)
    assert torch.allclose(attention_scores_fixed(Q, K), expected, atol=1e-5)


def test_mask_blocks_future_positions(qk):
    Q, K = qk
    mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
    scores = attention_scores_fixed(Q, K, mask=mask)
    assert torch.isneginf(scores[..., 0, -1]).all()
    assert torch.isfinite(scores[..., -1, 0]).all()


def test_masked_softmax_gives_zero_attention_to_future(qk):
    Q, K = qk
    mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
    weights = torch.softmax(attention_scores_fixed(Q, K, mask=mask), dim=-1)
    assert weights[..., 0, 1:] == pytest.approx(torch.zeros(B, H, T - 1).numpy(), abs=1e-6)
    assert weights.sum(-1) == pytest.approx(torch.ones(B, H, T).numpy(), abs=1e-5)


---
# Tier E — Software engineering in an ML lab


### E1. Deep config merge (no mutation)


In [ ]:
def deep_merge(base: dict, override: dict) -> dict:
    """Recursive merge; override wins. Do not mutate inputs."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — E1 deep config merge


@pytest.fixture
def base():
    return {"a": 1, "b": {"c": 2, "d": 3}}


def test_overrides_nested_values(base):
    assert deep_merge(base, {"b": {"c": 9}}) == {"a": 1, "b": {"c": 9, "d": 3}}


def test_adds_new_keys(base):
    assert deep_merge(base, {"e": 5})["e"] == 5


def test_does_not_mutate_inputs(base):
    snapshot = copy.deepcopy(base)
    override = {"b": {"c": 9}, "e": 5}
    deep_merge(base, override)
    assert base == snapshot
    assert override == {"b": {"c": 9}, "e": 5}


def test_returned_nested_dicts_are_not_shared(base):
    merged = deep_merge(base, {"b": {"c": 9}})
    merged["b"]["d"] = 99
    assert base["b"]["d"] == 3


def test_scalar_override_replaces_dict(base):
    assert deep_merge(base, {"b": 7}) == {"a": 1, "b": 7}


def test_empty_override_returns_equal_config(base):
    assert deep_merge(base, {}) == base


### E2. Deterministic seed helper
Be ready to say what this does **not** guarantee.


In [ ]:
def seed_everything(seed: int):
    """Seed random, numpy, torch; set cudnn deterministic flags when available."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — E2 deterministic seeding


def test_numpy_draws_are_reproducible():
    seed_everything(42)
    first = np.random.rand(3)
    seed_everything(42)
    assert np.random.rand(3) == pytest.approx(first)


def test_torch_draws_are_reproducible():
    seed_everything(42)
    first = torch.randn(3)
    seed_everything(42)
    assert torch.allclose(torch.randn(3), first)


def test_python_random_is_reproducible():
    seed_everything(42)
    first = [random.random() for _ in range(3)]
    seed_everything(42)
    assert [random.random() for _ in range(3)] == pytest.approx(first)


def test_different_seeds_give_different_draws():
    seed_everything(0)
    first = np.random.rand(5)
    seed_everything(1)
    assert not np.allclose(np.random.rand(5), first)


def test_model_initialisation_is_reproducible():
    seed_everything(7)
    first = nn.Linear(4, 4).weight.detach().clone()
    seed_everything(7)
    assert torch.allclose(nn.Linear(4, 4).weight.detach(), first)


### E3. Write the tests yourself (pytest fundamentals)
Practice the parts Qualcomm's software-engineering round probes: **fixtures**, **`parametrize`**,
**`monkeypatch`**, and `tmp_path`.

Using the helper below, write a `%%ipytest` cell that covers:
1. a fixture producing a reusable config dict
2. a `@pytest.mark.parametrize` case set for `resolve_artifact_root` defaults
3. a `monkeypatch.setenv` test proving the env var wins over the default
4. a `tmp_path` test that writes and reads a checkpoint file
5. a `pytest.raises` test for invalid input


In [ ]:
import os


def resolve_artifact_root(default: str = "./artifacts") -> Path:
    """Return the artifact root: ARTIFACT_ROOT env var if set, else `default`.

    Target for the monkeypatch exercise below.
    """
    if not default:
        raise ValueError("default must be a non-empty path")
    return Path(os.environ.get("ARTIFACT_ROOT", default))


In [ ]:
%%ipytest -qq
# Your tests — E3. Cover fixture / parametrize / monkeypatch / tmp_path / raises.


@pytest.fixture
def config():
    # YOUR CODE HERE
    raise NotImplementedError


def test_resolve_artifact_root_uses_default():
    # YOUR CODE HERE
    raise NotImplementedError


def test_env_var_overrides_default(monkeypatch):
    # YOUR CODE HERE: monkeypatch.setenv("ARTIFACT_ROOT", ...)
    raise NotImplementedError


def test_checkpoint_round_trip(tmp_path):
    # YOUR CODE HERE: torch.save / torch.load under tmp_path
    raise NotImplementedError


def test_empty_default_raises():
    # YOUR CODE HERE: pytest.raises(ValueError)
    raise NotImplementedError


### E4. Artifact path helper


In [ ]:
def artifact_path(root, model_name, version, split):
    """Return Path like root/model_name/v{version}/{split}.pt"""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — E4 artifact paths


def test_expected_layout():
    path = artifact_path("/tmp/artifacts", "resnet", 3, "best")
    assert Path(path).as_posix().endswith("resnet/v3/best.pt")


@pytest.mark.parametrize("version", [1, 7, 42])
def test_version_is_prefixed_with_v(version):
    assert f"v{version}" in Path(artifact_path("/root", "m", version, "best")).parts


def test_accepts_path_object_as_root(tmp_path):
    assert Path(artifact_path(tmp_path, "resnet", 1, "last")).is_relative_to(tmp_path)


def test_split_becomes_the_filename():
    assert Path(artifact_path("/root", "m", 1, "train")).name == "train.pt"


def test_different_versions_do_not_collide():
    first = Path(artifact_path("/root", "m", 1, "best"))
    second = Path(artifact_path("/root", "m", 2, "best"))
    assert first != second


---
# Tier F — Stretch (Physical AI / systems)


### F1. Occupancy grid from (x, y) points


In [ ]:
def occupancy_grid(points_xy, x_min, x_max, y_min, y_max, nx, ny):
    """points_xy: (N,2). Return (ny, nx) int grid counts. Ignore OOB points."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — F1 occupancy grid
# Convention: grid[row, col] == grid[y_bin, x_bin]


def test_shape_is_ny_by_nx():
    grid = occupancy_grid(np.zeros((0, 2)), 0, 1, 0, 1, nx=4, ny=2)
    assert grid.shape == (2, 4)


def test_counts_only_in_bounds_points():
    points = np.array([[0.1, 0.1], [0.9, 0.1], [2.0, 2.0]])
    grid = occupancy_grid(points, 0, 1, 0, 1, nx=2, ny=2)
    assert grid.sum() == 2


def test_points_land_in_expected_cells():
    points = np.array([[0.1, 0.1], [0.9, 0.1]])
    grid = occupancy_grid(points, 0, 1, 0, 1, nx=2, ny=2)
    assert grid[0, 0] == 1
    assert grid[0, 1] == 1
    assert grid[1].sum() == 0


def test_upper_bound_point_stays_inside_grid():
    grid = occupancy_grid(np.array([[1.0, 1.0]]), 0, 1, 0, 1, nx=2, ny=2)
    assert grid.sum() <= 1
    assert grid.shape == (2, 2)


def test_duplicate_points_accumulate():
    points = np.array([[0.25, 0.25]] * 3)
    grid = occupancy_grid(points, 0, 1, 0, 1, nx=2, ny=2)
    assert grid.max() == 3


def test_single_cell_grid_counts_everything_in_bounds():
    points = np.array([[0.1, 0.2], [0.7, 0.9]])
    grid = occupancy_grid(points, 0, 1, 0, 1, nx=1, ny=1)
    assert grid.tolist() == [[2]]


### F2. Naive voxel downsample


In [ ]:
def voxel_downsample(points, voxel_size):
    """points (N,3). Keep first point in each voxel. Return (M,3)."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# Unit tests — F2 voxel downsample


@pytest.fixture
def points():
    return np.array([[0.0, 0.0, 0.0], [0.1, 0.1, 0.1], [1.5, 0.0, 0.0]])


def test_merges_points_inside_one_voxel(points):
    assert voxel_downsample(points, voxel_size=1.0).shape[0] == 2


def test_small_voxel_keeps_all_points(points):
    assert voxel_downsample(points, voxel_size=0.01).shape[0] == len(points)


def test_large_voxel_collapses_to_single_point(points):
    assert voxel_downsample(points, voxel_size=100.0).shape[0] == 1


def test_output_is_a_subset_of_the_input(points):
    kept = voxel_downsample(points, voxel_size=1.0)
    assert all(any(np.allclose(p, q) for q in points) for p in kept)


def test_output_keeps_three_columns(points):
    assert voxel_downsample(points, voxel_size=1.0).shape[1] == 3


def test_is_idempotent(points):
    once = voxel_downsample(points, voxel_size=1.0)
    assert voxel_downsample(once, voxel_size=1.0).shape == once.shape


---
# 5-day sprint

| Day | Focus | Problems |
|-----|--------|----------|
| 1 | Metrics & losses | Warm-up, A1–A4 |
| 2 | Quantization | A6–A8, B6 + SQNR |
| 3 | PyTorch | B1–B5 |
| 4 | CV / efficiency | A5, C1, C3, C5 |
| 5 | Debug + SE | D1–D5, E1–E3 |

**Daily ritual:** 2 timed problems (20 min, narrate) + 1 untimed clean reference.

## What “good” looks like
1. Correctness first, then vectorization
2. Explicit edge cases
3. Readable names
4. Talk complexity / memory / numerics
5. Connect to deployment when relevant


---
# Reference solutions
Spoilers below — attempt the drills first.


In [ ]:
# === REFERENCE SOLUTIONS ===

def precision_recall_f1_sol(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if precision + recall > 0 else 0.0
    return float(precision), float(recall), float(f1)


def softmax_sol(logits):
    x = np.asarray(logits, dtype=float)
    if x.ndim == 1:
        z = x - x.max()
        e = np.exp(z)
        return e / e.sum()
    z = x - x.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def binary_cross_entropy_sol(y_true, y_prob, eps=1e-7):
    y = np.asarray(y_true, dtype=float)
    p = np.clip(np.asarray(y_prob, dtype=float), eps, 1 - eps)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def cross_entropy_sol(y_true, logits):
    logits = np.asarray(logits, dtype=float)
    y = np.asarray(y_true)
    z = logits - logits.max(axis=1, keepdims=True)
    log_probs = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
    return float(-log_probs[np.arange(len(y)), y].mean())


def confusion_counts_sol(y_true, y_pred, n_classes):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    C = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        C[t, p] += 1
    return C


def top_k_accuracy_sol(logits, y_true, k=5):
    logits = np.asarray(logits)
    y_true = np.asarray(y_true)
    k = min(k, logits.shape[1])
    topk = np.argpartition(-logits, kth=k - 1, axis=1)[:, :k]
    return float((topk == y_true[:, None]).any(axis=1).mean())


def binary_iou_sol(mask_true, mask_pred):
    a = np.asarray(mask_true).astype(bool)
    b = np.asarray(mask_pred).astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter / union) if union > 0 else 0.0


def box_iou_sol(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return float(inter / union) if union > 0 else 0.0


def exponential_moving_average_sol(values, momentum=0.9):
    out, ema = [], None
    for x in values:
        ema = x if ema is None else momentum * ema + (1 - momentum) * x
        out.append(ema)
    return out


def quantize_affine_sol(x, scale, zero_point, qmin=-128, qmax=127):
    q = np.round(np.asarray(x, dtype=float) / scale) + zero_point
    return np.clip(q, qmin, qmax).astype(np.int32)


def dequantize_affine_sol(q, scale, zero_point):
    return scale * (np.asarray(q, dtype=float) - zero_point)


def calibration_params_sol(x_min, x_max, qmin=0, qmax=255):
    if x_max == x_min:
        x_max = x_min + 1e-8
    scale = (x_max - x_min) / (qmax - qmin)
    zp = int(np.clip(np.round(qmin - x_min / scale), qmin, qmax))
    return float(scale), zp


print("Reference helpers loaded. Copy into exercise cells as needed.")


In [ ]:
# More reference solutions (Tier B–F)

class MLPClassifierSol(nn.Module):
    def __init__(self, in_dim, hidden, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x):
        return self.net(x)


def sgd_step_sol(params, grads, lr, weight_decay=0.0):
    updated = []
    for p, g in zip(params, grads):
        g = g + weight_decay * p
        updated.append(p - lr * g)
    return updated


class SlidingWindowDatasetSol(Dataset):
    def __init__(self, signal, window):
        self.signal = np.asarray(signal)
        self.window = window

    def __len__(self):
        return max(0, len(self.signal) - self.window + 1)

    def __getitem__(self, idx):
        return torch.as_tensor(self.signal[idx: idx + self.window])


def causal_mask_sol(seq_len):
    return np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)


def key_padding_mask_sol(lengths, max_len):
    lengths = np.asarray(lengths)
    return np.arange(max_len)[None, :] >= lengths[:, None]


def collect_activations_sol(model, x, layer_names):
    outs, handles = {}, []

    def make_hook(name):
        def hook(_m, _inp, out):
            outs[name] = out.detach().cpu()
        return hook

    modules = dict(model.named_modules())
    for name in layer_names:
        handles.append(modules[name].register_forward_hook(make_hook(name)))
    try:
        model(x)
    finally:
        for h in handles:
            h.remove()
    return outs


class MinMaxObserverSol:
    def __init__(self):
        self.min_val = None
        self.max_val = None

    def update(self, x):
        x = x.detach()
        mn, mx = float(x.min()), float(x.max())
        self.min_val = mn if self.min_val is None else min(self.min_val, mn)
        self.max_val = mx if self.max_val is None else max(self.max_val, mx)

    def compute_qparams(self, qmin, qmax):
        return calibration_params_sol(self.min_val, self.max_val, qmin, qmax)


def pairwise_l2_sol(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    aa = (a * a).sum(axis=1, keepdims=True)
    bb = (b * b).sum(axis=1, keepdims=True).T
    d2 = np.maximum(aa + bb - 2 * a @ b.T, 0.0)
    return np.sqrt(d2)


def one_hot_sol(indices, n_classes):
    idx = np.asarray(indices)
    out = np.zeros((len(idx), n_classes), dtype=float)
    out[np.arange(len(idx)), idx] = 1.0
    return out


def gather_rows_sol(mat, indices):
    mat = np.asarray(mat)
    indices = np.asarray(indices)
    return mat[np.arange(len(indices)), indices]


def nms_sol(boxes, scores, iou_threshold=0.5):
    boxes = np.asarray(boxes, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0:
        i = int(order[0])
        keep.append(i)
        if order.size == 1:
            break
        rest = order[1:]
        ious = np.array([box_iou_sol(boxes[i], boxes[j]) for j in rest])
        order = rest[ious <= iou_threshold]
    return keep


def streaming_mean_std_sol(stream):
    n, mean, M2 = 0, 0.0, 0.0
    for x in stream:
        n += 1
        delta = x - mean
        mean += delta / n
        M2 += delta * (x - mean)
    if n < 2:
        return mean, 0.0
    return mean, math.sqrt(M2 / (n - 1))


def decode_many_sol(jpeg_bytes_list, max_workers=4):
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        return list(pool.map(decode_jpeg, jpeg_bytes_list))


def deep_merge_sol(base, override):
    out = copy.deepcopy(base)
    for k, v in override.items():
        if k in out and isinstance(out[k], dict) and isinstance(v, dict):
            out[k] = deep_merge_sol(out[k], v)
        else:
            out[k] = copy.deepcopy(v)
    return out


def seed_everything_sol(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def artifact_path_sol(root, model_name, version, split):
    return Path(root) / model_name / f"v{version}" / f"{split}.pt"


def occupancy_grid_sol(points_xy, x_min, x_max, y_min, y_max, nx, ny):
    pts = np.asarray(points_xy, dtype=float)
    grid = np.zeros((ny, nx), dtype=int)
    for x, y in pts:
        if not (x_min <= x < x_max and y_min <= y < y_max):
            continue
        ix = int((x - x_min) / (x_max - x_min) * nx)
        iy = int((y - y_min) / (y_max - y_min) * ny)
        ix = min(ix, nx - 1)
        iy = min(iy, ny - 1)
        grid[iy, ix] += 1
    return grid


def voxel_downsample_sol(points, voxel_size):
    pts = np.asarray(points, dtype=float)
    seen, out = set(), []
    for p in pts:
        key = tuple(np.floor(p / voxel_size).astype(int))
        if key not in seen:
            seen.add(key)
            out.append(p)
    return np.asarray(out)


print("All reference solutions defined.")
